In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as signal
from scipy.signal import windows
from remezlp import *

try:
    import mpld3
    mpld3.enable_notebook()
    print("Modul mpld3 je instaliran.")
except:
    print("Modul mpld3 nije instaliran.")


    

In [ ]:
# crta spektar signala centriranog oko nule 
def plotSpectrum(x, xlabel = "", ylabel="", title="", img_name = "spektar.png", show=True):
    n = len(x)
    
    # np.fft.fftshift   => -fs/2 do fs/2 spektar 
    # np.fft.fftfreq(n) => niz frekvencija koje odgovaraju signalu duzine n 
    # np.fft.fftshift   => generise shift  shift (|x|)
    plt.plot(np.fft.fftshift(np.fft.fftfreq(n)),np.fft.fftshift(abs(np.fft.fft(x))));
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title);
    plt.savefig(img_name)
    if show:
        plt.show()
    
def stemSpectrum(x, xlabel = "", ylabel="", title="", img_name = "", show=True):
    n = len(x)
    F = np.fft.fftshift(np.fft.fftfreq(n))
    X = np.fft.fftshift(abs(np.fft.fft(x)))
    plt.stem(F, X)#, use_line_collection=True);
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title);
    plt.savefig(img_name)
    if show:
        plt.show()

def plotSignal(x, xlabel = "", ylabel="", title="", img_name = "", show=True):
    plt.plot(x);
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title);
    plt.savefig(img_name)
    if show:
        plt.show()

def stemSignal(x, xlabel = "", ylabel="", title="", img_name = "", show=True):
    plt.stem(x)# ,use_line_collection=True);
    plt.xlabel(xlabel); plt.ylabel(ylabel); plt.title(title);
    plt.savefig(img_name)
    if show:
        plt.show()
    

In [ ]:
def readSamples(fileName):
    samples = []
    inFile = open(fileName, "r")
    for line in inFile:
        samples.append( complex(line) )
    return np.array(samples)

# |X(f)| spektar

In [ ]:
fs = 61.44e6
testsignal = readSamples("testsignal.txt")
N = len(testsignal)
x = np.arange(N)
plotSpectrum(x, r"$f[MHz]$", r"|X|", "Spektar Signala")

freqs = (np.arange(N)/N - 0.5) * fs/1e6
plt.plot(freqs, np.fft.fftshift(abs(np.fft.fft(testsignal))))
plt.xlabel("f [MHz]")
plt.ylabel("|X(f)|");
plt.grid(True, linestyle="--", alpha=0.6)
plt.savefig("in_f_grid.png")

# Jitter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Postavke za opseg frekvencija (1 MHz do 1 GHz)
f = np.logspace(6, 9, 1000)

# Vrednosti džitera
jitters = [ (2.0e-12, '2.0 ps'), (1.0e-12, '1.0 ps'), (0.5e-12, '0.5 ps'), (0.2e-12, '0.2 ps'), (0.1e-12, '0.1 ps')]

# Rezolucije DAC-a u bitima
bits = [8, 10, 12, 14, 16]

fig, ax = plt.subplots(figsize=(8, 5.5), dpi=300)

# 1. Horizontalne linije za SQNR (kvantizacija)
for b in bits:
    sqnr = 6.02 * b + 1.76
    ax.axhline(y=sqnr, color='black', linestyle='--', linewidth=1.2, zorder=1)
    # Tekst sa belom pozadinom na ~2 MHz
    ax.text(2.2e6, sqnr, f'{b}b', verticalalignment='center', horizontalalignment='center',
            bbox=dict(facecolor='white', edgecolor='none', pad=2), fontsize=10, zorder=3)

# 2. Dijagonalne linije za SNR usled džitera 
for sigma, label in jitters:
    snr_j = -20 * np.log10(2 * np.pi * f * sigma)
    ax.plot(f, snr_j, linewidth=1.8, zorder=2, label=f'$t_j = {label}$')
    
    # Text na samim linijama
    f_ref = 1e8
    snr_ref = -20 * np.log10(2 * np.pi * f_ref * sigma)
    if 40 <= snr_ref <= 110:
        ax.text(f_ref * 1.08, snr_ref - 1.2, label, 
                rotation=-29, verticalalignment='center', horizontalalignment='left',
                bbox=dict(facecolor='white', edgecolor='none', pad=0.8), fontsize=9.5)

# Ose i granice
ax.set_xscale('log')
ax.set_xlim(1e6, 1e9)
ax.set_ylim(40, 110)

ax.set_xlabel('$f$ [Hz]', fontsize=11)
ax.set_ylabel('$SNR$ [dB]', fontsize=11)
ax.set_title('Degradacija $SNR$ usled džitera', fontsize=14, pad=15, color='#283593', loc='center')

# Dodavanje legende
ax.legend(loc='upper right', frameon=True, facecolor='white', framealpha=0.9, fontsize=9.5)

# Mreža (grid)
ax.grid(True, which='both', color='lightgrey', linestyle='-', linewidth=0.6)

plt.tight_layout()
plt.savefig('degradacija_snr_jitter.png', dpi=300)
plt.show()

# Digitalni spektar (Ekspander)

In [ ]:
fs = 61.44 
testsignal = readSamples("testsignal.txt")
N = len(testsignal)

# Normalizovana učestanost F od -0.5 do 0.5 
F = np.arange(N) / N - 0.5
X_abs = np.fft.fftshift(abs(np.fft.fft(testsignal)))

#plt.figure(figsize=(7, 4))
plt.plot(F, X_abs)
plt.xlabel(r"$F = \frac{f}{f_s}$")
plt.ylabel("|X|")
plt.xlim(-0.5, 0.5)

#plt.grid(True)
plt.savefig("in_F.pdf")
plt.show()


In [ ]:
def plot_spectre(Fpass, Fstop, testsignal, save_path, L = 2):
    
    testsignal_up = np.zeros(len(testsignal) * L, dtype=complex)
    testsignal_up[::L] = testsignal

    N_up = len(testsignal_up)

    
    # Normalizovana učestanost F od -0.5 do 0.5 za novi signal
    F_up = np.arange(N_up) / N_up - 0.5
    X_up_abs = np.fft.fftshift(abs(np.fft.fft(testsignal_up)))

    plt.plot(F_up, X_up_abs)

    xticks = [-0.5, 0, 0.5]
    xticklabels = ["-0.5", "0", "0.5"]
    plt.xticks(xticks, xticklabels)

    plt.axvline(Fpass, color='g', linestyle='--', linewidth=1, label=f"$F_{{pass}} = {Fpass:.4f}$")
    plt.axvline(-Fpass, color='g', linestyle='--', linewidth=1)
    plt.axvline(Fstop, color='r', linestyle='--', linewidth=1, label=f"$F_{{stop}} = {Fstop:.4f}$")
    plt.axvline(-Fstop, color='r', linestyle='--', linewidth=1)
   
    plt.xlabel(r"$F = \frac{f}{f_s}$")
    plt.ylabel("|X(F)|")
    plt.xlim(-0.5, 0.5)
    plt.savefig(save_path + ".png")
    plt.show()

In [ ]:
def exp_plot_filter(Fpass, Fstop, testsignal, save_path, L=2):
    # 1. Slabljenje
    AdB = 60.0
    deltaPass = 10.0 ** (-AdB / 20.0)
    deltaStop = deltaPass

    # 2. Nadodabiranje
    oversampled_signal = np.zeros(len(testsignal) * L, dtype=complex)
    oversampled_signal[::L] = testsignal

    N_up = len(oversampled_signal)
    F_axis = np.arange(N_up) / N_up - 0.5

    # 3. Projektovanje i filtriranje
    h = remezlp(Fpass, Fstop, deltaPass, deltaStop, nPoints=8192 * 8, Nmax=500)

    delay = len(h) // 2
    oversampled_signal_perext = np.concatenate((oversampled_signal, oversampled_signal[0:delay]))
    xi = signal.lfilter(h, 1, oversampled_signal_perext)
    xi = xi[delay : delay + len(oversampled_signal)]

    N_xi = len(xi)
    F_axis_xi = np.arange(N_xi) / N_xi - 0.5

    # 4. Prikaz side-by-side (1 red, 2 kolone)
    fig, axes = plt.subplots(1, 2, figsize = (7, 4))

    # Levi grafik - pre filtriranja
    axes[0].plot(F_axis, np.fft.fftshift(abs(np.fft.fft(oversampled_signal))))
    axes[0].plot([-0.5, -Fstop, -Fpass, Fpass, Fstop, 0.5], [0, 0, 1, 1, 0, 0], 'r')
    axes[0].set_xticks([-0.5, 0, 0.5])
    axes[0].set_xticklabels(["-0.5", "0", "0.5"])
    axes[0].axvline(Fpass, color='g', linestyle='--', linewidth=1, label=f"$F_{{pass}} = {Fpass:.4f}$")
    axes[0].axvline(-Fpass, color='g', linestyle='--', linewidth=1)
    axes[0].axvline(Fstop, color='r', linestyle='--', linewidth=1, label=f"$F_{{stop}} = {Fstop:.4f}$")
    axes[0].axvline(-Fstop, color='r', linestyle='--', linewidth=1)
    axes[0].set_xlabel(r"$F$")
    axes[0].set_ylabel("|X(F)|")
    axes[0].set_title("Spektar pre filtriranja")

    # Desni grafik - posle filtriranja
    axes[1].plot(F_axis_xi, np.fft.fftshift(abs(np.fft.fft(xi))))
    axes[1].plot([-0.5, -Fstop, -Fpass, Fpass, Fstop, 0.5], [0, 0, 1, 1, 0, 0], 'r')
    axes[1].set_xticks([-0.5, 0, 0.5])
    axes[1].set_xticklabels(["-0.5", "0", "0.5"])
    axes[1].set_xlabel(r"$F$")
    axes[1].set_ylabel(r"$|X(F)|$")
    axes[1].set_title("Spektar posle filtriranja")

    plt.tight_layout()
    plt.savefig(save_path + ".png")
    plt.show()

    return xi

In [ ]:
from remezlp import *
L = 2  # Faktor ekspanzije

fmax = 25.0e6
fs = 61.44e6

# Prvi stepen  
Fmax1 = fmax / fs
Fpass1 = Fmax1 / L
Fstop1 = (1 - Fmax1) / L 

test_signal = readSamples("testsignal.txt")

# ove 2 funkcije kopiramo u sva 3 ostala stepena 
plot_spectre(Fpass1, Fstop1, test_signal,"M21_pre")
xi1 = exp_plot_filter(Fpass1, Fstop1, test_signal, "M21_spojeno")



In [ ]:
# Drugi stepen 
Fmax2  = Fmax1 / L  
Fpass2 = Fmax2 / L
Fstop2 = (1 - Fmax2) / L 

plot_spectre(Fpass2, Fstop2, xi1,"M22_pre")
xi2 = exp_plot_filter(Fpass2, Fstop2, xi1, "M22_spojeno")
print(Fpass2)


In [ ]:
# Treci stepen 

Fmax3  = Fmax2 / L  
Fpass3 = Fmax3 / L
Fstop3 = (1 - Fmax3) / L 

plot_spectre(Fpass3, Fstop3, xi2,"M23_pre")
xi3 = exp_plot_filter(Fpass3, Fstop3, xi2, "M23_spojeno")
print(Fpass3)


In [ ]:
# Cetvrti stepen 

Fmax4  = Fmax3 / L  
Fpass4 = Fmax4 / L
Fstop4 = (1 - Fmax4) / L 

plot_spectre(Fpass4, Fstop4, xi3, "M24_pre")
xi4 = exp_plot_filter(Fpass4, Fstop4, xi3, "M24_spojeno")

print(Fpass4)

In [ ]:
L = 16  # Faktor ekspanzije
fmax = 25
fs = 61.44  # Ukupna učestanost odabiranja

# Ekspanzija
testsignal_up = np.zeros(len(testsignal) * L, dtype=complex)
testsignal_up[::L] = testsignal

N_up = len(testsignal_up)
F_up = np.arange(N_up) / N_up - 0.5
X_up_abs = np.fft.fftshift(abs(np.fft.fft(testsignal_up)))

# Tačne granice
F_max = fmax / fs          # 25 / 61.44 ≈ 0.4069
F_pass = F_max / L         # ≈ 0.02543 (kraj korisnog signala)
F_stop = (1 - F_max) / L   # ≈ 0.03707 (početak prve replike)

plt.plot(F_up, X_up_abs, label="|X(F)|", color='tab:blue')



# prikazuju tačno 0, F_pass, F_stop i 1/16 na X-osi
plt.xticks(
    [0, F_pass, F_stop, 1/L],
    ['0', r'$F_{\mathrm{pass}}$', r'$F_{\mathrm{stop}}$', r'$\frac{1}{16}$'],
    fontsize=11
)

# Zumiranje
plt.xlim(-0.001, 0.08)

plt.xlabel(r"$F = \frac{f}{16f_{s}}$", fontsize=11)
plt.ylabel("|X(F)|")

plt.savefig("zoom_16_dir.pdf", bbox_inches='tight')
plt.show()


# Poluopsežni filtar generisanje grafika

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt 

signal_len = 1000
dots_len   = 1000 
max_range  = 1 / 2 

H_b = np.ones(signal_len)

idx = np.linspace(0, 1, dots_len) - 1 / 2 

shift = max_range / 2
H_b[(idx > shift) |  (idx < -shift)] = 0 

plt.figure()
plt.xlim(-max_range, max_range)
plt.xlabel(r"$ -\frac{1}{2} {\leq} F {\leq} \frac{1}{2}$")
plt.ylabel(r"$H_{b}(F)$")
plt.plot(idx, H_b)
plt.savefig("Hb_filter.png")
plt.show()

# Kompenzacija $\frac{sin(x)}{x}$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

signal_len  = 1000
Ts          = signal_len / 2

t           = np.arange(-signal_len/2, signal_len)

h_t         = np.ones(int(2 * signal_len -Ts))

mask = (t < Ts) & (t > 0)


h_t_T       = np.ones(int(2 * signal_len - Ts))
h_t_T[mask] = 0 


h_final = h_t - h_t_T 

#plt.figure(figsize=(8, 4))
plt.plot(t, h_final, linewidth=2)

plt.ylim(-0.2, 1.2)
plt.xlabel("t")
plt.ylabel(r"$g_{0}(t)$")
plt.title("Kolo zadrške nultog reda (NRZ)")

plt.savefig("NRZ.pdf")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import *

fmax = 0.4 # max. frequency of interest
n = 9 # number of points for approximation

f = fmax * np.arange(0, n+1, 1)/n

H_sinc = np.sin(np.pi*f)/(np.pi*f+1e-15)
H_sinc[0] = 1.0

f_goal = np.zeros(2*(n+1))
f_goal[0::2] = f
f_goal[1::2] = f+1e-3
f_goal *= 2	# In firls function Nyquist frequency is 1

goal = np.zeros(2*(n+1))
goal[0::2] = 1.0/H_sinc
goal[1::2] = 1.0/H_sinc

h_invsinc = firls(n, f_goal, goal)
w,H = freqz(tuple(h_invsinc),tuple([1]))
H_sinc = np.sin(w/2)/(w/2+1e-15)
H_sinc[0] = 1.0

plt.plot(w/(2*np.pi),np.absolute(1.0/H_sinc), 'k', aa=True, linewidth=2.5,label=r'$x/\sin(x)$')
plt.plot(w/(2*np.pi),np.absolute(H), 'gray', aa=True, linewidth=2.5,label=r'$|H(\mathrm{j} F)|$')
plt.legend(loc='upper left')
df = 0.005

for i in range(0,len(goal),2):
    plt.plot((f_goal[i]/2-df,f_goal[i+1]/2+df),[goal[i], goal[i+1]], 'r',aa=True, linewidth=2.5)
plt.xlim(0,0.5)
plt.ylim(1,1.4)

#plt.grid()
plt.xlabel(r'$F$')
plt.ylabel(r'$|H(\mathrm{j} F)|$')
plt.tight_layout()
plt.savefig("sinc_kompenzacija.png")
plt.show()

In [ ]:
# Normalizuje maksimalnu vrenost sinc filtra na 1 
x = -1.0 * np.ones(n)
x[0::2] = 1

h_invsinc_norm = h_invsinc / np.sum(h_invsinc*x)
print("Normalized sinc correction filter")

print(h_invsinc_norm)
plt.stem(h_invsinc_norm)
plt.show()
w,H = freqz(tuple(h_invsinc_norm),tuple([1]))

plt.plot(w/(2*np.pi),20*np.log10(np.absolute(H)), 'k', aa=True, linewidth=2.5)
plt.xlabel(r'$F$')
plt.ylabel(r'$|H_\mathrm{norm}(\mathrm{j} F)|~\mathrm{dB}$')
plt.grid()
plt.xlim(0,0.5)
plt.ylim(-5,0)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import *
logScale = False
def fn_nrz(f):
    T = 1
    i=0
    a = np.pi*f*T
    v = np.sin(a+1e-15)/(a+1e-15)
    if logScale:
        g = 20*np.log10(abs(v))
    else:
        g = abs(v)
    return g
f = np.arange(0,0.5,0.005)
H_dac = fn_nrz(f) 

plt.plot(f, 20*np.log10(H_dac), '--k', linewidth=2.5,aa=True, label=r'$H_\mathrm{NRZ}$')
h_invsinc = [0.00170705, -0.00582203, 0.01783758, -0.06830803, 0.81265063, 
             -0.06830803, 0.01783758, -0.00582203, 0.00170705]
w, H_invsinc = freqz(h_invsinc, 1, 2*np.pi*f)
H_invsinc_dB = 20*np.log10(np.absolute(H_invsinc))

plt.plot(f, H_invsinc_dB, ':k', linewidth=2.5,aa=True, label=r'$H_\mathrm{invsinc}$')
H_total = H_dac * H_invsinc
H_total_dB = 20*np.log10(np.absolute(H_total))


plt.plot(f, H_total_dB, '-k', linewidth=2.5,aa=True, label=r'$H_\mathrm{NRZ} H_\mathrm{invsinc}$')


# Određivanje DC nivoa (vrednost na f=0) oko kog tražimo +/-0.025 dB
dc_level = H_total_dB[0]

# Crtanje linije maksimalne učestanosti
plt.axvline(0.4, color='r', linestyle='--', linewidth=1.5, label=r'$F_\mathrm{max} = 0.4$')

# Crtanje granica +/- 0.025 dB oko DC nivoa
plt.axhline(dc_level + 0.025, color='g', linestyle=':', linewidth=1.5, label=r'$\pm0.025\,\mathrm{dB}$')
plt.axhline(dc_level - 0.025, color='g', linestyle=':', linewidth=1.5)

plt.xlabel(r'$f/f_\mathrm{s}$')
plt.ylabel(r'$|H(\mathrm{j}\omega)|~\mathrm{[dB]}$')
plt.legend(loc="lower left")

# Uveličavanje:
plt.xlim(0, 0.4)                  # Uveličava na opseg od interesa [0, 0.4]
plt.ylim(dc_level - 0.08, dc_level + 0.08) # Uveličava na opseg Y ose oko DC nivoa

plt.grid(True)
plt.tight_layout()
plt.savefig("zadovoljava.pdf")
plt.show()


# <center> Druga faza Projekta (Koeficijenti filtra) </center>

In [ ]:
from scipy.signal import *
from remezhb import *

In [ ]:
# poluopsežni filtar u diskretnom vremenskom domenu 

import matplotlib.pyplot as plt
import numpy as np

data_len = 1000
n = np.linspace(-20, 20, data_len)
x = np.zeros(data_len)

# np.where(uslov, vrednost ako je uslov ispunjen, vrednost ako uslov nije ispunjen )
x = np.where(n == 0, 0.5, 0.5 * np.sin((n * np.pi) / 2) / ((n * np.pi) / 2))

plt.xlabel(r"$n$")
plt.ylabel(r"$h(n)$")
plt.xlim(-10, 10)
plt.plot(n, x)

plt.savefig("polu.png")
plt.show()


In [ ]:
def filterSignal(Fpass, AdB, expand_factor, filter_name, print_info):
    h = remezhb(Fpass, AdB)
    
    if print_info == True: 
        print(filter_name)
        print(h)
        print()
    
    hmz = np.zeros(len(h) * expand_factor)
    hmz[::expand_factor] = h
 
    w, H = freqz(h, 1, worN=1024, fs=1)
 
    return hmz, w, H

In [ ]:
def plot_individual_filters(Fpasses, ws, Hs, AdB=60):
    m1, m2, m3, m4 = 8, 4, 2, 1
 
    for i in range(len(ws)):
        w = ws[i]
        H = Hs[i]
        Fpass = Fpasses[i]
 
        plt.figure()
 
        plt.plot(w, 20.0 * np.log10(abs(H)), 'k', linewidth=1.5, aa=True)
        plt.axvline(Fpass, color='g', linestyle='--', linewidth=1)
        plt.axvline(0.5 - Fpass, color='r', linestyle='--', linewidth=1)
        plt.xlabel(r'$F$')
        plt.ylabel(r'$|H_{%d}(e^{j2\pi F})|$ [dB]' % (i + 1))
        plt.title('Stepen ' + str(i + 1))
        plt.ylim(-80.0, 5.0)
        plt.xlim(0, 0.5)
 
        plt.tight_layout()
        plt.savefig('H' + str(i + 1) + '_filter.png')
        plt.show()


def plot_total_freq_response(w, Hmz): 
    plt.figure()
 
    plt.plot(w, 20.0 * np.log10(abs(Hmz)), 'k', linewidth=1.5, aa=True)
    plt.xlabel(r'$F$')
    plt.ylabel(r'$|H(e^{j 2 \pi F})|$ [dB]')
 
    plt.ylim(-60.0, 5.0)
    plt.xlim(0, 0.5)
 
    plt.tight_layout()
    plt.savefig('total_filter.png', dpi=150)
    plt.show()


In [ ]:
def final_filter_spectre(pass_frequencies, plot, print_info, AdB=60):
    # H1(z) = H1 , H2(z) = H2 itd
    # H1 -> [2] -> H2 -> [2] -> H3 -> [2] -> H4 -> [2]
    # [16] -> H1(z**8) * H2(z**4) * H3(z ** 2) * H4(z)
    expand_factor = 8
    m1, m2, m3, m4 = 8, 4, 2, 1
    Fpass1, Fpass2, Fpass3, Fpass4 = pass_frequencies
 
    # h1 filtriranje
    hmz1, w1, H1 = filterSignal(Fpass1, AdB, m1, "H1", print_info)
 
    # h2 filtriranje
    hmz2, w2, H2 = filterSignal(Fpass2, AdB, m2, "H2", print_info)
 
    # h3 filtriranje
    hmz3, w3, H3 = filterSignal(Fpass3, AdB, m3, "H3", print_info)
 
    # h4 filtriranje
    hmz4, w4, H4 = filterSignal(Fpass4, AdB, m4, "H4", print_info)

 
    h_conv1 = np.convolve(hmz1, hmz2)
    h_conv2 = np.convolve(hmz3, hmz4)
    hmz = np.convolve(h_conv1, h_conv2)
 
    w, Hmz = freqz(hmz, 1, worN=1024, fs=1)
 
    if plot == True: 
        # crtanje 
        ws = [w1, w2, w3, w4]
        Hs = [H1, H2, H3, H4]

        plot_individual_filters(pass_frequencies, ws, Hs, AdB )
        plot_total_freq_response(w, Hmz)
    
    return hmz


In [ ]:
#Interpolacija 

pass_frequencies = (0.20345, 0.12172, 0.0508, 0.02543)

# signal nakon interpolacije 
hmz              = final_filter_spectre(pass_frequencies, False, True)





# <center> Cordic Algoritam implementacija(uzeto iz sveske) </center>

In [ ]:
import math
import numpy
from numpy import pi

###############
# Functions
###############
class CORDIC():
    def __init__(self, n, coord, mode):
        """
        n - number of iterations
        coord - CORDIC coordinate system
              1 - polar
              0 - linear
             -1 - hyperbolic
        mode - "rotation" for CORDIC in rotation mode,
               "vectoring" for CORDIC in vectoring mode
        """
        self.n = n
        self.setCoordinateSystem(coord)
        self.setMode(mode)
        self.kn = self.calculateKn()

    def setCoordinateSystem(self, coord):
        if coord == "polar":
            self.m = 1
        elif coord == "linear":
            self.m = 0
        elif coord == "hyperbolic":
            self.m = -1
        else:
            raise ValueError("Unsupported coordinate system.")

    def setMode(self, mode):
        if mode!="rotation" and mode!="vectoring":
            raise ValueError("Unsupported mode.")
        self.mode = mode
        
    def calculate(self,x0, y0, z0):
        """
        Calculate n iterations of CORDIC algorithm.
        x0, y0, z0 - inputs
        Returns (xn, yn, zn)
        """
        n = self.n
        m = self.m
        mode = self.mode
        sgn = lambda a: (a>=0) - (a<0)
        x = x0
        y = y0
        z = z0
        z = z%(2*pi)
        if m==1 and mode=="rotation":
            # Prerotation for rotation mode
            if z>pi/2 and z<=3*pi/2:
                x = -x
                y = -y
                z = z - pi
                sign = -1
            elif z>3*pi/2:
                z = z-3*pi/2
                x0 = y
                y = -x
                x = x0
        elif m==1 and mode=="vectoring":
            # Prerotation for vectoring mode
            if x<0:
                x = -x
                y = -y
                z = z0 + pi
        else:
            pass
        repeatedIteration = 4
        if m==1:
            nstart = 0
        else:
            nstart = 1
        for i in range(nstart, n):
            if m == -1:
                if i==repeatedIteration:
                    repeatedIteration = 3*repeatedIteration+1
                    repetitions = 2
                else:
                    repetitions = 1
            else:
                repetitions = 1
            for _ in range(repetitions):
                if mode=="rotation":
                    sigma = sgn(z)
                else:
                    sigma = -sgn(y)
                xnew = x - m*sigma*y*2**(-i)
                ynew = y + sigma*x*2**(-i)
                x = xnew
                y = ynew

                if m==1:
                    z = z - sigma * math.atan(2**(-i))
                elif m==0:
                    z = z - sigma * 2**(-i)
                else:
                    z = z - sigma * math.atanh(2**(-i))
        return (xnew, ynew, z)

    def calculateKn(self):
        """
        Calculate the overall scaling factor for n iterations
        """
        if self.m == 0:
            return 1.0
        k = 1
        if self.m==1:
            nstart = 0
        else:
            nstart = 1
        repeatedIteration = 4
        n = self.n
        for i in range(nstart,n):
            if self.m == -1:
                if i==repeatedIteration:
                    repeatedIteration = 3*repeatedIteration+1
                    repetitions = 2
                else:
                    repetitions = 1
            else:
                repetitions = 1
            for _ in range(repetitions):    
                k *= 1.0/math.sqrt(1.0+self.m*2.0**(-2*i))
        return k

    def __call__(self, x, y, z):
        """
        Overload of built-in call method
        """
        return self.calculate(x, y, z)
    
    def sincos(self, z):
        """
        Method to calculate sin(z) and cos(z) by using CORDIC 
        in polar coordinates and rotation mode.        
        """
        if type(z) is not numpy.ndarray:
            zarray = numpy.array([z])
        else:
            zarray = z
        n = len(zarray)
        xarray = numpy.zeros(n)
        yarray = numpy.zeros(n)
        for i in range(n):
            x = self.kn
            y = 0
            z = zarray[i]
            x, y, z = self.__call__(x, y, float(z))
            xarray[i] = x
            yarray[i] = y
        return (xarray, yarray)
    
    def magang(self, x, y):
        """
        Method to calculate sqrt(x^2+y^2) and atan(y/x) by using CORDIC 
        in polar coordinates and vectoring mode.        
        """
        if type(x) is not numpy.ndarray:
            xarray = numpy.array([x])
            yarray = y
        else:
            xarray = x
            yarray = y
        n = len(xarray)
        mag_array = numpy.zeros(n)
        ang_array = numpy.zeros(n)
        for i in range(n):
            x = xarray[i]
            y = yarray[i]
            mag, y, ang = self.__call__(float(x), float(y), 0)
            mag_array[i] = mag
            ang_array[i] = ang
        return (mag_array, ang_array)
    

In [ ]:
import matplotlib.pyplot as plt

it     = 10 
M      = 4096
P      = 127
angle  = (P / M) * 2 * pi 
A      = 100  
n      = np.arange(M)

# pozivanje cordic algoritma 
cordic = CORDIC(10, "polar", "rotation")

theta = n * angle
cos_theta, sin_theta=cordic.sincos(theta)

out_signal = A * (cos_theta + 1j * sin_theta) 

#plt.plot(theta, cos_theta, 'r')
#plt.plot(theta, sin_theta, 'b')

plt.plot(n, out_signal)
plt.show()

In [ ]:
# 1. Izračunaj FFT i uradi shift da nula bude u sredini
fft_signal = np.fft.fft(out_signal)
fft_shifted = np.fft.fftshift(fft_signal)

# 2. Frekvencijska osa i njen shift
freqs = np.fft.fftfreq(M)
freqs_shifted = np.fft.fftshift(freqs)

# 3. Amplitudski spektar (normalizovan sa M)
fft_mag = np.abs(fft_shifted) / M

# 4. Crtanje spektra
plt.figure()
plt.plot(freqs_shifted, fft_mag)
plt.title("Amplitudski spektar")
plt.xlabel(r"$F$")
plt.ylabel(r"$|X(F)|$")
#plt.grid(True)
plt.savefig("CORDIC_pravilan.png")
plt.show()

# <center> Model Celog sistema </center>

In [ ]:
class PlotConfig():  
    def __init__(self, plot_F=False, plot_f=False, img_num_F=0, img_num_f=0, dont_save=True):
        self.plot_F = plot_F
        self.plot_f = plot_f
        self.img_num_F = img_num_F
        self.img_num_f = img_num_f
        self.dont_save = dont_save

In [ ]:
def plot_one_interp_level_F(xi, i, dont_save):
    N_xi = len(xi)
    F_axis_xi = np.arange(N_xi) / N_xi - 0.5

    fig, ax = plt.subplots()
    ax.plot(F_axis_xi, np.fft.fftshift(abs(np.fft.fft(xi))))
    ax.set_xticks([-0.5, 0, 0.5])
    ax.set_xticklabels(["-0.5", "0", "0.5"])
    ax.set_xlabel(r"$F$")
    ax.set_ylabel(r"$|X(F)|$")
    ax.set_title("Spektar nakon filtriranja")

    if not dont_save:
        plt.savefig("interp_F" + str(i) + ".png")

    plt.tight_layout()
    plt.show()


def plot_one_interp_level_f(xi, i, dont_save, fs_current):
    N_xi = len(xi)
    freqs_mhz = (np.arange(N_xi) / N_xi - 0.5) * (fs_current / 1e6)
    f_max_mhz = (fs_current / 2) / 1e6

    fig, ax = plt.subplots()
    ax.plot(freqs_mhz, np.fft.fftshift(abs(np.fft.fft(xi))))
    ax.set_xticks([-f_max_mhz, 0, f_max_mhz])
    ax.set_xticklabels([f"{-f_max_mhz:.2f}", "0", f"{f_max_mhz:.2f}"])
    ax.set_xlabel(r"$f$ [MHz]")
    ax.set_ylabel(r"$|X(f)|$")
    ax.set_title("Spektar nakon filtriranja")
    plt.grid(True, linestyle="--", alpha=0.6)

    if not dont_save:
        plt.savefig("interp_f" + str(i) + ".png")

    plt.tight_layout()
    plt.show()


def interp_one_level(in_sig, Fpass, Fstop, L):
    AdB = 60.0
    deltaPass = 10.0 ** (-AdB / 20.0)
    deltaStop = deltaPass

    # Nadodabiranje
    oversampled_signal = np.zeros(len(in_sig) * L, dtype=complex)
    oversampled_signal[::L] = in_sig

    # Projektovanje i filtriranje
    h = remezlp(Fpass, Fstop, deltaPass, deltaStop, nPoints=8192 * 8, Nmax=500)

    delay = len(h) // 2
    oversampled_signal_perext = np.concatenate((oversampled_signal, oversampled_signal[0:delay]))
    xi = signal.lfilter(h, 1, oversampled_signal_perext)
    xi = xi[delay: delay + len(oversampled_signal)]

    return xi


def interpolate(filter_stages, testsignal, plot_cfg=None, fs_in_def=61.44e6, L=2):
    if plot_cfg is None:
        plot_cfg = PlotConfig()

    in_sig = testsignal
    fs_in = fs_in_def

    for Fpass, Fstop in filter_stages:
        xi = interp_one_level(in_sig, Fpass, Fstop, L)
        fs_in *= 2

        if plot_cfg.plot_F:
            plot_one_interp_level_F(xi, plot_cfg.img_num_F, plot_cfg.dont_save)
        if plot_cfg.plot_f:
            plot_one_interp_level_f(xi, plot_cfg.img_num_f, plot_cfg.dont_save, fs_in)

        plot_cfg.img_num_F += 1
        plot_cfg.img_num_f += 1

        in_sig = xi

    return xi

In [ ]:
def complex_shift(input_sig, omega, fs_dac, i, show=True, dont_save = False):
    n = np.arange(len(input_sig))

    cordic = CORDIC(10, "polar", "rotation")
    theta = omega * n

    cos_n, sin_n = cordic.sincos(theta)
    complex_sig = cos_n + 1j * sin_n

    out_sig = np.real(complex_sig * input_sig)

    # Plotovanje spektra u odnosu na fs_dac (u MHz)
    L_sig = len(out_sig)
    X_out = np.fft.fftshift(np.fft.fft(out_sig))
    X_out_dB = 20 * np.log10(np.abs(X_out) + 1e-12)
    X_out_dB -= np.max(X_out_dB)  # Normalizacija na 0 dB

    # Frekvencijska osa u MHz (-fs_dac/2 do +fs_dac/2)
    freqs_mhz = (np.arange(L_sig) / L_sig - 0.5) * (fs_dac / 1e6)

    fig = plt.figure(figsize=(8, 3.5))
    plt.plot(freqs_mhz, X_out_dB)
    plt.ylim(-80, 5)
    plt.xlim(-fs_dac / 2e6, fs_dac / 2e6)
    plt.xlabel(r"$f\text{ [MHz]}$")
    plt.ylabel(r"$|X(f)|\text{ [dB]}$")
    plt.title("Signal nakon kompleksnog mešača")
    plt.grid(True, linestyle="--", alpha=0.6)
    
    if dont_save == False: plt.savefig(f"cordic_{i}.png", bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return out_sig

In [ ]:
def inv_sinc_filter(n, Fmax):
    F = Fmax * np.arange(0, n + 1, 1) / n

    H_sinc = np.sin(np.pi * F) / (np.pi * F + 1e-15)
    H_sinc[0] = 1.0

    F_goal = np.zeros(2 * (n + 1))
    F_goal[0::2] = F
    F_goal[1::2] = F + 1e-3
    F_goal *= 2  # Za firls Nyquist frekvencija je 1

    goal = np.zeros(2 * (n + 1))
    goal[0::2] = 1.0 / H_sinc
    goal[1::2] = 1.0 / H_sinc

    h_invsinc = firls(n, F_goal, goal)
    return h_invsinc


def correct_output(n, Fmax, cordic_out, fs_dac, i, show=True, dont_save = False):
    h_invsinc = inv_sinc_filter(n, Fmax)
    filtered_sig = signal.lfilter(h_invsinc, 1.0, cordic_out)

    # Crtanje spektra u odnosu na fs_dac (u MHz)
    L_sig = len(filtered_sig)
    X_filtered = np.fft.fftshift(np.fft.fft(filtered_sig))
    X_filtered_dB = 20 * np.log10(np.abs(X_filtered) + 1e-12)
    X_filtered_dB -= np.max(X_filtered_dB)

    # Frekvencijska osa u MHz (-fs_dac/2 do +fs_dac/2)
    freqs_mhz = (np.arange(L_sig) / L_sig - 0.5) * (fs_dac / 1e6)

    fig = plt.figure()
    plt.plot(freqs_mhz, X_filtered_dB)
    plt.ylim(-80, 5)
    plt.xlim(-fs_dac / 2e6, fs_dac / 2e6)
    plt.xlabel(r"$f\text{ [MHz]}$")
    plt.ylabel(r"$|X(f)|\text{ [dB]}$")
    plt.title("Signal nakon inverznog sinc filtra")
    plt.grid(True, linestyle="--", alpha=0.6)
    
    if dont_save == False : plt.savefig(f"inv_{i}.png", bbox_inches="tight")

    if show:
      plt.show()
      
    else:
        plt.close(fig) 
        
    return filtered_sig

In [ ]:
def nrz_reconstruction(xr, NyqZones, fs_dac, i, show=True, dont_save = False):
    nrz = np.ones(NyqZones)
    xr_upsampled = np.zeros((len(xr) - 1) * NyqZones + 1)
    xr_upsampled[::NyqZones] = xr
    xr_nrz = np.convolve(nrz, xr_upsampled)

    L_nrz = len(xr_nrz)
    XrnrzdB = 20 * np.log10(np.fft.fftshift(np.abs(np.fft.fft(xr_nrz))) + 1e-12)
    XrnrzdB -= np.max(XrnrzdB)

    freqs_nrz = (np.arange(L_nrz) / L_nrz - 0.5) * (fs_dac * NyqZones) / 1e6

    fig = plt.figure() 

    plt.plot(freqs_nrz, XrnrzdB)
    plt.ylim(-80, 5)
    plt.xlim(-NyqZones * fs_dac / 2e6, NyqZones * fs_dac / 2e6)
    
    plt.xlabel(r"$f\text{ [MHz]}$")
    plt.ylabel(r"$|X(f)|\text{ [dB]}$")
    plt.title( "NRZ rekonstrukcija" )
    
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    
    if dont_save == False: plt.savefig(f"nrz_{i}.png")

    if show:
        plt.show()
    else:
        plt.close(fig) 

def rf_reconstruction(xr, NyqZones, fs_dac, i, show=True, dont_save = False):
    rf = np.concatenate([np.ones(int(NyqZones / 2)), -1.0 * np.ones(int(NyqZones / 2))])
    xr_upsampled = np.zeros((len(xr) - 1) * NyqZones + 1)
    xr_upsampled[::NyqZones] = xr
    xr_rf = np.convolve(rf, xr_upsampled)

    L_rf = len(xr_rf)
    XrrfdB = 20 * np.log10(np.fft.fftshift(np.abs(np.fft.fft(xr_rf))) + 1e-12)
    XrrfdB -= np.max(XrrfdB)

    freqs_rf = (np.arange(L_rf) / L_rf - 0.5) * (fs_dac * NyqZones) / 1e6

    fig = plt.figure()
    
    plt.plot(freqs_rf, XrrfdB)
    plt.ylim(-80, 5)
    plt.xlim(-NyqZones * fs_dac / 2e6, NyqZones * fs_dac / 2e6)
    plt.xlabel(r"$f\text{ [MHz]}$")
    plt.ylabel(r"$|X(f)|\text{ [dB]}$")
    plt.title("Rf rekonstrukcija")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    
    if dont_save == False: plt.savefig(f"rf_{i}.png")


    if show:
        plt.show()
    else:
        plt.close(fig) 



In [ ]:
def plot_nrz_vs_rf_comparison(xr, NyqZones, fs_dac, i=0, f_shift=None, show=True, dont_save=False):
  if dont_save and not show:
    return

  # 1. Upsampling sa nulama
  xr_upsampled = np.zeros((len(xr) - 1) * NyqZones + 1)
  xr_upsampled[::NyqZones] = xr

  # 2. NRZ (Zero-Order Hold) impuls
  nrz = np.ones(NyqZones)
  xr_nrz = np.convolve(nrz, xr_upsampled)
  L_nrz = len(xr_nrz)
  X_nrz_dB = 20 * np.log10( np.abs(np.fft.fftshift(np.fft.fft(xr_nrz))) + 1e-12 )
  X_nrz_dB -= np.max(X_nrz_dB)  # normalizacija na 0 dB

  # 3. RZ
  half_zone = max(1, int(NyqZones / 2))
  rf = np.concatenate([np.ones(half_zone), -1.0 * np.ones(half_zone)])
  xr_rf = np.convolve(rf, xr_upsampled)
  L_rf = len(xr_rf)
  X_rf_dB = 20 * np.log10(np.abs(np.fft.fftshift(np.fft.fft(xr_rf))) + 1e-12)
  X_rf_dB -= np.max(X_rf_dB)  # normalizacija na 0 dB

  # 4. Frekvencijska osa u MHz
  freqs_mhz = (np.arange(L_nrz) / L_nrz - 0.5) * (fs_dac * NyqZones) / 1e6

  # 5. Crtanje
  fig, ax = plt.subplots(dpi=150)

  ax.plot(freqs_mhz, X_nrz_dB, color='b', linewidth=1.4, label='NRZ')
  ax.plot(freqs_mhz, X_rf_dB, color='#F87171', linestyle='--', linewidth=1.4, label='RZ')

  fn_mhz = fs_dac / 2e6  # Nyquist-ova frekvencija (fs_dac / 2)

  ax.set_ylim(-80, 5)
  ax.set_xlim(-NyqZones * fn_mhz, NyqZones * fn_mhz)
  ax.set_xlabel(r'$f\ \mathrm{[MHz]}$', fontsize=11)
  ax.set_ylabel(r'$|X(f)|\ \mathrm{[dB]}$', fontsize=11)

  title_str = 'Poređenje NRZ i RZ rekonstrukcije'
  if f_shift is not None:
    title_str += f' ($f_{{shift}} = {f_shift/1e6:.1f}\\ \\mathrm{{MHz}}$)'
  ax.set_title(title_str, fontsize=12)

  ax.legend(loc='upper right', framealpha=0.9)
  ax.grid(True, linestyle='--', alpha=0.5)

  plt.tight_layout()

  if not dont_save:
    fig.savefig(f'nrz_vs_rz_{i}.png', bbox_inches='tight')

  if show:
    plt.show()

  plt.close(fig)  

In [ ]:
Fpass_list = np.array([0.20345, 0.12172, 0.0508, 0.02543])
Fstop_list = np.array([0.29655, 0.37828, 0.4492, 0.47457])

n = 9
i = 0

Fmax = 0.4
Nyq_zones = 4
dont_save = False 

reconstruction = "nrz"

# Ulazni signal i interpolacija
testsignal = readSamples("testsignal.txt")

N = len(testsignal)
M = 16

# Frekvencije
fs = 61.44e6  # Ulazni sample rate
fs_dac = M * fs  # DAC sample rate = 983.04 MHz
fshift = np.array([120e6, 240e6, 400e6, 600e6, 700e6])


# Poziv interpolacije 
filter_stages = list(zip(Fpass_list, Fstop_list))
plot_cfg0 = PlotConfig(plot_F=False, plot_f=True, img_num_f=0, img_num_F=0, dont_save=False)

xi = interpolate(filter_stages, testsignal, plot_cfg0, fs, L=2)


# omega se racuna u odnosu na fs_dac
omega = 2 * np.pi * fshift / fs_dac

print("P vrednosti:", (fshift * N) / fs)

out_signals, filtered_signals = [], []

for i, om in enumerate(omega):
  # 1. CORDIC pomeraj
  out_sig = complex_shift(xi, om, fs_dac, i, show=True, dont_save = dont_save)

  # 2. Korekcija inverznim sinc filtrom
  filtered_sig = correct_output(n, Fmax, out_sig, fs_dac, i, show=True, dont_save = dont_save)

  # 3. Rekonstrukcija (NRZ ili RZ)
  if reconstruction == "nrz":
    nrz_reconstruction(filtered_sig, Nyq_zones, fs_dac, i, show=True, dont_save = dont_save)
  elif reconstruction == "rz":
    rf_reconstruction(filtered_sig, Nyq_zones, fs_dac,  i, show=True, dont_save = dont_save)

  filtered_signals.append(filtered_sig)
  out_signals.append(out_sig)

# <center> 2.4 **Korišćenje projektovanih funkcija sistema** </center>

In [ ]:
Fpass_list = np.array([0.20345, 0.12172, 0.0508, 0.02543])
Fstop_list = np.array([0.29655, 0.37828, 0.4492, 0.47457])

n = 9
i = 15
Fmax = 0.4
Nyq_zones = 4
dont_save = False 


M = 16 
fs = 61.44e6  # Ulazni sample rate
fs_dac = M * fs 
f_shift_nrz = 3 * M * fs / 8 

omega = 2 * np.pi * f_shift_nrz / fs_dac

# Ulazni signal i interpolacija
testsignal = readSamples("testsignal.txt")
N = len(testsignal)
M = 16

filter_stages = list(zip(Fpass_list, Fstop_list))
plot_cfg1 = PlotConfig(plot_F=False, plot_f=True, img_num_f=0, img_num_F=0, dont_save=False)

xi = interpolate(filter_stages, testsignal, plot_cfg1, fs, L=2)




# 1. kompleksni shift
out_sig = complex_shift(xi, omega, fs_dac, i, show=True, dont_save = dont_save)

# 2. Korekcija inverznim sinc filtrom
filtered_sig = correct_output(n, Fmax, out_sig, fs_dac, i, show=True, dont_save = dont_save)

# 3. Rekonstrukcija (NRZ)
nrz_reconstruction(filtered_sig, Nyq_zones, fs_dac, i, show=True, dont_save = dont_save)


plot_nrz_vs_rf_comparison(filtered_sig, Nyq_zones, fs_dac, i, f_shift_nrz, True, dont_save = dont_save)


In [ ]:
Fpass_list = np.array([0.20345, 0.12172, 0.0508, 0.02543])
Fstop_list = np.array([0.29655, 0.37828, 0.4492, 0.47457])
n = 9
i = 16
Fmax = 0.4
Nyq_zones = 4
dont_save = False


M       = 16 
fs      = 61.44e6  # Ulazni sample rate
fs_dac  = M * fs 
f_nyq_2 = 860.16e6
f_shift_rf = fs_dac - f_nyq_2

omega = 2 * np.pi * f_shift_rf / fs_dac


# Ulazni signal i interpolacija
testsignal      = readSamples("testsignal.txt")
N               = len(testsignal)

filter_stages   = list(zip(Fpass_list, Fstop_list))
plot_cfg2       = PlotConfig(plot_F=False, plot_f=True, img_num_f=0, img_num_F=0, dont_save=False)
xi              = interpolate(filter_stages, testsignal, plot_cfg2, fs, L=2)

out_sig = complex_shift(xi, omega, fs_dac, i, show=True, dont_save = dont_save)

# 2. Rekonstrukcija (RF)
rf_reconstruction(out_sig, Nyq_zones, fs_dac, i, show=True, dont_save = dont_save)


plot_nrz_vs_rf_comparison(filtered_sig, Nyq_zones, fs_dac, i, f_shift_rf, True, dont_save = dont_save)
